# Preprocessing

In [ ]:
# Imports and paths
import sys
sys.path.append('../')

import matplotlib.pyplot as plt
import wfdb

from src.preprocessing.label_utils import load_all_labels, SUPERCLASSES
from src.preprocessing.preprocess  import preprocess_record, bandpass_filter, normalize_signal
from src.preprocessing.dataset     import ECGDataset
from torch.utils.data import DataLoader
from src.utils.config import CFG

DATA_PATH = CFG['data']['path']

In [2]:
# Load labels and print class distribution
Y = load_all_labels(
    db_path  = DATA_PATH + 'ptbxl_database.csv',
    scp_path = DATA_PATH + 'scp_statements.csv'
)
print(Y[['age', 'sex', 'superclass', 'strat_fold']].head(10))

Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
         age  sex superclass  strat_fold
ecg_id                                  
1       56.0    1     [NORM]           3
2       19.0    0     [NORM]           2
3       37.0    1     [NORM]           5
4       24.0    0     [NORM]           3
5       19.0    1     [NORM]           4
6       18.0    1     [NORM]           4
7       54.0    0     [NORM]           7
8       48.0    0       [MI]           9
9       55.0    0     [NORM]          10
10      22.0    1     [NORM]           9


In [ ]:
# Visualize preprocessing effect on ONE signal
sample_row  = Y.iloc[0]
raw, meta   = wfdb.rdsamp(DATA_PATH + sample_row['filename_lr'])

filtered    = bandpass_filter(raw)
normalized  = normalize_signal(filtered)

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
lead = 1  # Lead II — most commonly shown in clinical settings

axes[0].plot(raw[:, lead],        color='gray')
axes[0].set_title('Raw Signal — Lead II')
axes[0].set_ylabel('mV')

axes[1].plot(filtered[:, lead],   color='steelblue')
axes[1].set_title('After Bandpass Filter (0.5–40 Hz)')
axes[1].set_ylabel('mV')

axes[2].plot(normalized[:, lead], color='darkgreen')
axes[2].set_title('After Normalization (zero mean, unit variance)')
axes[2].set_ylabel('Normalized')
axes[2].set_xlabel('Time steps (100 = 1 second)')

plt.tight_layout()
plt.savefig(CFG['paths']['results'] + 'preprocessing_comparison.png', dpi=150)
plt.show()
print(f"Label for this record: {sample_row['superclass']}")

In [ ]:
# Show windowing
windows = preprocess_record(raw)
print(f"One 10-sec record → {len(windows)} windows")
print(f"Each window shape: {windows[0].shape}  (time_steps, leads)")

fig, axes = plt.subplots(len(windows), 1, figsize=(14, 14))
for i, w in enumerate(windows):
    axes[i].plot(w[:, lead], linewidth=0.8)
    axes[i].set_title(f'Window {i+1}  (samples {i*CFG["data"]["stride"]} – {i*CFG["data"]["stride"]+CFG["data"]["window_size"]})')
    axes[i].set_yticks([])
plt.suptitle('7 overlapping windows from one 10-second ECG', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Build train/val/test datasets and verify DataLoader
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]
test_df  = Y[Y.strat_fold == 10]

print(f"Train records: {len(train_df)}  → ~{len(train_df)*7} windows")
print(f"Val records:   {len(val_df)}    → ~{len(val_df)*7} windows")
print(f"Test records:  {len(test_df)}   → ~{len(test_df)*7} windows")

# Build dataset (preload=False to be safe on memory)
train_ds = ECGDataset(train_df, DATA_PATH, preload=False)
val_ds   = ECGDataset(val_df,   DATA_PATH, preload=False)
test_ds  = ECGDataset(test_df,  DATA_PATH, preload=False)

# Verify one batch loads correctly
loader = DataLoader(train_ds, batch_size=CFG['training']['batch_size_full'], shuffle=True, num_workers=CFG['training']['num_workers'])
x_batch, y_batch = next(iter(loader))

print(f"\nBatch shapes:")
print(f"  x: {x_batch.shape}  ← (batch, leads, time) = (32, 12, 250)")
print(f"  y: {y_batch.shape}  ← (batch, classes)     = (32, 5)")
print(f"\nExample label vector: {y_batch[0].numpy()}")
print(f"Means: {SUPERCLASSES}")
print(f"  → this record has: {[SUPERCLASSES[i] for i,v in enumerate(y_batch[0]) if v==1]}")